# RAG: Question-Answers on Private Document

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

In [ ]:
# pip install -q pypdf
# pip install -q wikipedia
!pip install pypdf

In [ ]:
def load_documents(file_path):
    from langchain_community.document_loaders.pdf import PyPDFLoader
    loader = PyPDFLoader(file_path)
    data =  loader.load()
    return data


data = load_documents("/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/projects/RAG/us_constitution.pdf")

# print(data[0].page_content)
# print(data[0].metadata)
print(f"Total Pages: {len(data)}")

In [ ]:

def load_documents(file_path):
    import os 
    from langchain_community.document_loaders import (Docx2txtLoader,PyPDFLoader, TextLoader)

    name, ext = os.path.splitext(file_path)

    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
        data =  loader.load()
    elif ext == ".docx":
        loader = Docx2txtLoader(file_path)
        data =  loader.load()
    elif ext == ".txt":
        loader = TextLoader(file_path)
        data =  loader.load()
    else:
        print("Unsupported file format")
        return None
    return data


def load_wikipedia_data(query):
    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query, load_max_docs=1)
    data = loader.load()
    return data


## creating chunks

In [ ]:
def chunk_data(data, chunk_size=512, chunk_overlap=50):
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, 
        chunk_overlap=300,
        separators=["\n\n", "\n", ".", ""]
    )
    return text_splitter.split_documents(data)

# calculating Cost

In [ ]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')
    

## Asking and Getting Answers

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI


def ask_and_get_answer(vector_store, q, k=3):
    from langchain_classic.chains.retrieval_qa.base import RetrievalQA
    # from langchain_groq import ChatGroq


    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)
    # llm = ChatGroq(
    #     model="llama-3.3-70b-versatile",
    #     api_key= os.getenv("GROQ_API_KEY")
    # )
    retriever = vector_store.as_retriever(
        search_type='mmr', 
        search_kwargs={'k': 5, 'fetch_k': 10}
    )

    chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)
    answer = chain.invoke(q)
    return answer


# Using Chroma as a VectorDB

In [ ]:
pip install -q chromadb

In [26]:
def create_embedding_chroma(chunks, persist_directory="./chroma_db", collection_name="my_collection"):
    from langchain_chroma import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings

    

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = Chroma.from_documents(
        chunks, 
        embeddings, 
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    return vector_store


def load_embedding_chroma(persist_directory="./chroma_db", collection_name="my_collection"):
    from langchain_chroma import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings, collection_name=collection_name)
    return vector_store


In [ ]:
data = load_documents('/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/files/rag_powered_by_google_search.pdf')

chunks = chunk_data(data)
vector_store = create_embedding_chroma(chunks)

print(len(chunks))


In [ ]:
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

In [ ]:
print(answer['result'])

In [ ]:
db = load_embedding_chroma()
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

In [ ]:
print(answer['result'])

In [ ]:
docs = vector_store.similarity_search("8 million", k=3)
for d in docs:
    print(d.page_content)

In [ ]:
import re

for doc in data:
    # Remove null characters
    text = doc.page_content.replace("\x00", "")

    # Merge broken line breaks
    text = re.sub(r"\n(?=[a-z])", " ", text)  # join lowercase continuations
    text = re.sub(r"\n", " ", text)           # remove remaining newlines

    doc.page_content = text

In [ ]:

chunks = chunk_data(data)
vector_store = create_embedding_chroma(chunks)

print(len(chunks))

In [ ]:
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

In [ ]:
data1 = load_documents('/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/files/guidedread017.pdf')

chunks = chunk_data(data1)
vector_store = create_embedding_chroma(chunks)

print(len(chunks))


In [ ]:
q  = 'how many different type of dogs are there on the planet?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

# Adding Memory (chat History )

In [ ]:
import re

def clean_docs(docs):
    for doc in docs:
        # Join lines where a number/word is broken across lines
        doc.page_content = re.sub(r'(\w+)\n(\w+)', r'\1 \2', doc.page_content)
        # Collapse all remaining single newlines into spaces
        doc.page_content = doc.page_content.replace('\n', ' ').strip()
    return docs

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings


data = load_documents('/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/files/guidedread017.pdf')
data = clean_docs(data)
chunks = chunk_data(data)
vector_store = create_embedding_chroma(chunks)


print(len(chunks))


In [ ]:

from langchain_classic.chains import ConversationalRetrievalChain  # Import class for building conversational AI chains 
from langchain_classic.memory import ConversationBufferMemory  # Import memory for storing conversation history
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key= os.getenv("GROQ_API_KEY")
)


retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5}) 

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

crc = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    chain_type = 'stuff',
    verbose=True
)

In [ ]:
def ask_question(q, chain):
    result = chain.invoke({"question": q})
    return result['answer']

In [ ]:
q = 'how many different type of dogs are there on the planet?'
result = ask_question(q, crc)
print(result)

In [ ]:
q= "divide the number by 10 and give me the answer"
result = ask_question(q, crc)
print(result)

In [ ]:
q = "Multiply the number by 10 and give me the answer"
result = ask_question(q, crc)
print(result)

### Using a Custom Prompts

In [29]:

from langchain_classic.chains import ConversationalRetrievalChain  # Import class for building conversational AI chains 
from langchain_classic.memory import ConversationBufferMemory  # Import memory for storing conversation history
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key= os.getenv("GROQ_API_KEY")
)


retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5}) 

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


system_template = r'''
    use the following  pieces of contex to answer the user's question
    If you don't find the answer in the provide context, just say "Sorry, I don't know the answer to that question."
    -------------------
    context: ```{context}```
'''

user_template = r'''
    question: ```{question}```
'''

messages = [
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(user_template)
]

qa_prompt = ChatPromptTemplate.from_messages(messages)

crc = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    chain_type = 'stuff',
    combine_docs_chain_kwargs={'prompt': qa_prompt},
    verbose=True
)

In [25]:
print(qa_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    use the following  pieces of contex to answer the user\'s question\n    Before answering the translate your response to malayalam.\n    If you don\'t find the answer in the provide context, just say "Sorry, I don\'t know the answer to that question."\n    -------------------\n    context: ```{context}```\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\n    question: ```{question}```\n'), additional_kwargs={})]


In [27]:
db = load_embedding_chroma()
q = 'how many different type of dogs are there on the planet?'
result = ask_question(q, crc)
print(result)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 707.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
    use the following  pieces of contex to answer the user's question
    Before answering the translate your response to malayalam.
    If you don't find the answer in the provide context, just say "Sorry, I don't know the answer to that question."
    -------------------
  	 context: ```9	
-Dogs tend to have more than one puppy at a time and those puppies are called a litter. -There is said to be 400 million dogs on the planet. -Dogs have formed such strong bonds with humans they have earned the nickname “man’s best friend.”

  	
-Dogs tend to have more than one puppy at a time and those puppies are called a litter. -There is said to be 400 million dogs on the planet. -Dogs have formed such strong bonds with humans they have earned the nickname “man’s best friend.”

  	
-Dogs tend to have more than one puppy at a time and those puppies are called a litter. -There is said

In [30]:
q = "What is the capital of France?"
result = ask_question(q, crc)
print(result)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
    use the following  pieces of contex to answer the user's question
    If you don't find the answer in the provide context, just say "Sorry, I don't know the answer to that question."
    -------------------
    context: ```14

14

14

12

12```

Human: 
    question: ```What is the capital of France?```


> Finished chain.

> Finished chain.
Sorry, I don't know the answer to that question.


In [31]:
q = "what is  total type of dogs on the planet?"
result = ask_question(q, crc)
print(result)



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: What is the capital of France?
Assistant: Sorry, I don't know the answer to that question.
Follow Up Input: what is  total type of dogs on the planet?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
    use the following  pieces of contex to answer the user's question
    If you don't find the answer in the provide context, just say "Sorry, I don't know the answer to that question."
    -------------------
  	 context: ```9	
-Dogs tend to have more than one puppy at a time and those puppies are called a litter. -There is said to be 400 million dogs on the planet. -Dogs have formed such strong bonds with humans they have earned the nickname “man’s best frie